
# PySHRED on Lorenz-derived 50D observations (multiple seeds)

This notebook:
- generates data from a **3D Lorenz latent system** mapped to higher-dimensional observations,
- uses a **non-nested, 4-layer MLP mixer** (`nested=False`, `n_layers=4`),
- trains several PySHRED models with different random seeds,
- uses a **3D PySHRED bottleneck** (`hidden_size=3`),
- reports reconstruction metrics and linear alignment of learned latents to true Lorenz latents.


In [ ]:

import importlib
import subprocess
import sys


def ensure_module(module_name: str, pip_name: str):
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        print(f"Installing missing dependency: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])


# Needed by PySINDy on some environments (provides pkg_resources)
ensure_module("pkg_resources", "setuptools")
# PySHRED imports SINDy modules at import time
ensure_module("pysindy", "pysindy")

import numpy as np
import pandas as pd
import torch
from scipy.integrate import odeint
from scipy.signal import savgol_filter
from scipy.special import comb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

from pyshred import DataManager, SHRED, SHREDEngine, GRU, MLP, SINDy_Forecaster


ModuleNotFoundError: No module named 'pkg_resources'

## Data generator (adapted from your script)

In [ ]:

_MIXER_CACHE = {}


def library_size(n, poly_order, include_sine=False):
    total = 0
    for k in range(poly_order + 1):
        total += int(comb(n + k - 1, k, exact=True))
    if include_sine:
        total += n
    return total


def _mixer_cache_key(seed, n_points, max_layers, cond_thresh, leaky_alpha, add_bias):
    return (
        int(seed),
        int(n_points),
        int(max_layers),
        float(cond_thresh),
        float(leaky_alpha),
        bool(add_bias),
    )


def _col_normalize(W):
    return W / (np.linalg.norm(W, axis=0, keepdims=True) + 1e-12)


def _sample_well_conditioned_W(rng, dim=3, cond_thresh=50.0, max_tries=20000):
    for _ in range(max_tries):
        W = rng.uniform(-1.0, 1.0, size=(dim, dim)).astype(np.float32)
        W = _col_normalize(W).astype(np.float32)
        if np.linalg.cond(W) < cond_thresh:
            return W
    raise RuntimeError(f"Failed to sample W with cond < {cond_thresh}.")


def _get_or_create_nested_mixer(seed, n_points, max_layers, cond_thresh, leaky_alpha, add_bias):
    key = _mixer_cache_key(seed, n_points, max_layers, cond_thresh, leaky_alpha, add_bias)
    if key in _MIXER_CACHE:
        return _MIXER_CACHE[key]

    rng = np.random.RandomState(int(seed))
    Ws_full = [_sample_well_conditioned_W(rng, dim=3, cond_thresh=float(cond_thresh)) for _ in range(int(max_layers))]

    n = int(n_points)
    A = (rng.randn(n, 3).astype(np.float32) / np.sqrt(3.0)).astype(np.float32)
    b = rng.uniform(-0.5, 0.5, size=(n,)).astype(np.float32) if bool(add_bias) else np.zeros((n,), dtype=np.float32)

    mixer = {"Ws_full": Ws_full, "A": A, "b": b}
    _MIXER_CACHE[key] = mixer
    return mixer


def finite_diff_first(y, dt):
    dy = np.empty_like(y)
    dy[1:-1] = (y[2:] - y[:-2]) / (2.0 * dt)
    dy[0] = (y[1] - y[0]) / dt
    dy[-1] = (y[-1] - y[-2]) / dt
    return dy


def finite_diff_second(y, dt):
    ddy = np.empty_like(y)
    ddy[1:-1] = (y[2:] - 2.0 * y[1:-1] + y[:-2]) / (dt ** 2)
    ddy[0] = ddy[1]
    ddy[-1] = ddy[-2]
    return ddy


def maybe_smooth(y, window=11, poly=3):
    if window is None:
        return y
    T = y.shape[0]
    w = int(window)
    if w >= T:
        w = T - 1 if (T - 1) % 2 == 1 else T - 2
    if w < 5:
        return y
    if w % 2 == 0:
        w += 1
    return savgol_filter(y, window_length=w, polyorder=poly, axis=0, mode="interp")


def lorenz_coefficients(normalization, poly_order=3, sigma=10.0, beta=8/3, rho=28.0):
    Xi = np.zeros((library_size(3, poly_order), 3))
    Xi[1, 0] = -sigma
    Xi[2, 0] = sigma * normalization[0] / normalization[1]
    Xi[1, 1] = rho * normalization[1] / normalization[0]
    Xi[2, 1] = -1
    Xi[6, 1] = -normalization[1] / (normalization[0] * normalization[2])
    Xi[3, 2] = -beta
    Xi[5, 2] = normalization[2] / (normalization[0] * normalization[1])
    return Xi


def simulate_lorenz(z0, t, sigma=10.0, beta=8/3, rho=28.0):
    f = lambda z, tt: [sigma * (z[1] - z[0]), z[0] * (rho - z[2]) - z[1], z[0] * z[1] - beta * z[2]]
    df = lambda z, dz, tt: [
        sigma * (dz[1] - dz[0]),
        dz[0] * (rho - z[2]) + z[0] * (-dz[2]) - dz[1],
        dz[0] * z[1] + z[0] * dz[1] - beta * dz[2],
    ]

    z = odeint(f, z0, t)
    dt = t[1] - t[0]
    dz = np.zeros(z.shape)
    ddz = np.zeros(z.shape)
    for i in range(t.size):
        dz[i] = f(z[i], dt * i)
        ddz[i] = df(z[i], dz[i], dt * i)
    return z, dz, ddz


def generate_lorenz_data(
    ics,
    t,
    n_points,
    linear=True,
    normalization=None,
    sigma=10,
    beta=8/3,
    rho=28,
    seed=0,
    noise_strength=0.0,
    smooth=True,
    sg_window=11,
    sg_poly=3,
    cond_thresh=50.0,
    n_layers=4,
    leaky_alpha=0.2,
    add_bias=True,
    *,
    nested=False,
    max_layers=4,
):
    ics = np.asarray(ics, dtype=np.float64)
    t = np.asarray(t, dtype=np.float64)

    n_ics = ics.shape[0]
    n_steps = t.size
    if n_steps < 3:
        raise ValueError("Need at least 3 time points for finite differences.")
    dt = float(t[1] - t[0])

    z = np.zeros((n_ics, n_steps, 3), dtype=np.float32)
    dz = np.zeros_like(z)
    ddz = np.zeros_like(z)
    for i in range(n_ics):
        zi, dzi, ddzi = simulate_lorenz(ics[i], t, sigma=sigma, beta=beta, rho=rho)
        z[i] = zi.astype(np.float32)
        dz[i] = dzi.astype(np.float32)
        ddz[i] = ddzi.astype(np.float32)

    if normalization is not None:
        norm = np.asarray(normalization, dtype=np.float32).reshape((1, 1, 3))
        z *= norm
        dz *= norm
        ddz *= norm

    n_layers = int(n_layers)
    if n_layers < 1:
        raise ValueError("n_layers must be >= 1")
    if nested and int(max_layers) < n_layers:
        raise ValueError("max_layers must be >= n_layers when nested=True")

    n = int(n_points)
    if nested:
        mixer = _get_or_create_nested_mixer(seed, n_points, max_layers, cond_thresh, leaky_alpha, add_bias)
        Ws = mixer["Ws_full"][:n_layers]
        A = mixer["A"]
        b = mixer["b"]
    else:
        rng = np.random.RandomState(int(seed))
        Ws = [_sample_well_conditioned_W(rng, dim=3, cond_thresh=float(cond_thresh)) for _ in range(n_layers)]
        A = (rng.randn(n, 3).astype(np.float32) / np.sqrt(3.0)).astype(np.float32)
        b = rng.uniform(-0.5, 0.5, size=(n,)).astype(np.float32) if bool(add_bias) else np.zeros((n,), dtype=np.float32)

    def _leaky_relu(x):
        return np.where(x >= 0.0, x, float(leaky_alpha) * x)

    def h_mlp(z_batch_2d):
        xh = z_batch_2d
        for li, W in enumerate(Ws):
            xh = xh @ W.T
            if li < len(Ws) - 1:
                xh = _leaky_relu(xh)
        return xh

    x_clean = np.zeros((n_ics, n_steps, n), dtype=np.float32)
    for i in range(n_ics):
        hz = h_mlp(z[i])
        x_clean[i] = hz @ A.T + b

    rng_noise = np.random.RandomState(int(seed) + 12345)
    x_noisy = (x_clean + noise_strength * rng_noise.randn(*x_clean.shape).astype(np.float32)).astype(np.float32) if noise_strength and noise_strength > 0 else x_clean

    if smooth:
        x_used = np.empty_like(x_noisy)
        for i in range(n_ics):
            x_used[i] = maybe_smooth(x_noisy[i], window=sg_window, poly=sg_poly).astype(np.float32)
    else:
        x_used = x_noisy

    dx_used = np.empty_like(x_used)
    ddx_used = np.empty_like(x_used)
    for i in range(n_ics):
        dx_used[i] = finite_diff_first(x_used[i], dt).astype(np.float32)
        ddx_used[i] = finite_diff_second(x_used[i], dt).astype(np.float32)

    if normalization is None:
        sindy_coefficients = lorenz_coefficients([1, 1, 1], sigma=sigma, beta=beta, rho=rho)
    else:
        sindy_coefficients = lorenz_coefficients(np.asarray(normalization), sigma=sigma, beta=beta, rho=rho)

    data = {}
    data["t"] = t
    data["z"] = z
    data["dz"] = dz
    data["ddz"] = ddz
    data["sindy_coefficients"] = sindy_coefficients.astype(np.float32)

    data["x_nl"] = x_used
    data["dx_nl"] = dx_used
    data["ddx_nl"] = ddx_used

    data["x"] = x_used
    data["dx"] = dx_used
    data["ddx"] = ddx_used

    data["x_clean"] = x_clean
    data["x_noisy"] = x_noisy
    data["x_used"] = x_used

    data["mixing_seed"] = int(seed)
    data["mixing_n_layers"] = int(n_layers)
    data["mixing_nested"] = bool(nested)
    data["mixing_Ws"] = Ws
    data["mixing_A"] = A
    data["mixing_b"] = b

    if nested:
        data["mixing_Ws_full"] = _get_or_create_nested_mixer(seed, n_points, max_layers, cond_thresh, leaky_alpha, add_bias)["Ws_full"]

    return data


def get_lorenz_data(
    n_ics,
    noise_strength=0,
    smooth=True,
    sg_window=11,
    sg_poly=3,
    seed=0,
    input_dim=128,
    t_step=0.01,
    n_layers=4,
    cond_thresh=50.0,
    leaky_alpha=0.2,
    add_bias=True,
    nested=False,
    max_layers=4,
):
    t = np.arange(0, 5, t_step)
    ic_means = np.array([0, 0, 25])
    ic_widths = 2 * np.array([36, 48, 41])
    rng = np.random.RandomState(seed)
    ics = ic_widths * (rng.rand(n_ics, 3) - 0.5) + ic_means

    data = generate_lorenz_data(
        ics,
        t,
        n_points=input_dim,
        linear=False,
        normalization=np.array([1/40, 1/40, 1/40]),
        seed=seed,
        noise_strength=noise_strength,
        smooth=smooth,
        sg_window=sg_window,
        sg_poly=sg_poly,
        n_layers=int(n_layers),
        cond_thresh=float(cond_thresh),
        leaky_alpha=float(leaky_alpha),
        add_bias=bool(add_bias),
        nested=bool(nested),
        max_layers=int(max_layers),
    )

    data["x"] = data["x"].reshape((-1, input_dim))
    data["dx"] = data["dx"].reshape((-1, input_dim))
    data["ddx"] = data["ddx"].reshape((-1, input_dim))
    data["z_flat"] = data["z"].reshape((-1, 3))
    return data


## Generate Lorenz-derived observed data (non-nested, 4-layer mixer)

In [ ]:

data_seed = 7
input_dim = 50
n_ics = 12

data = get_lorenz_data(
    n_ics=n_ics,
    noise_strength=0.01,
    smooth=True,
    seed=data_seed,
    input_dim=input_dim,
    t_step=0.01,
    n_layers=4,
    nested=False,      # required non-nested setup
    max_layers=4,
)

X_obs = data["x"]       # shape (T_total, 50)
Z_true = data["z_flat"] # shape (T_total, 3)

pd.DataFrame({"X_rows": [X_obs.shape[0]], "X_dim": [X_obs.shape[1]], "Z_dim": [Z_true.shape[1]]})


## Train multiple PySHRED models (different seeds, bottleneck=3)

In [ ]:

train_seeds = [0, 1, 2]

results = []
trained = {}

for s in train_seeds:
    np.random.seed(s)
    torch.manual_seed(s)

    manager = DataManager(lags=20, train_size=0.7, val_size=0.15, test_size=0.15)

    # Use all 50 observed channels as sensor measurements.
    manager.add_data(
        data=X_obs,
        id=f"lorenz50_seed{s}",
        measurements=X_obs,
        compress=False,
    )

    train_ds, val_ds, test_ds = manager.prepare()

    seq = GRU(hidden_size=3, num_layers=1)  # bottleneck = 3
    dec = MLP(hidden_sizes=[128, 128], dropout=0.0)
    lf = SINDy_Forecaster(poly_order=2, include_sine=False, dt=0.01)

    shred = SHRED(sequence_model=seq, decoder_model=dec, latent_forecaster=lf)

    _ = shred.fit(
        train_dataset=train_ds,
        val_dataset=val_ds,
        batch_size=128,
        num_epochs=40,
        lr=1e-3,
        sindy_regularization=0.0,
        verbose=False,
        patience=8,
    )

    test_mse = shred.evaluate(test_ds, batch_size=128)

    engine = SHREDEngine(manager, shred)
    Z_hat = engine.sensor_to_latent(X_obs)

    # Linear alignment check: can learned latent map to true Lorenz latent via affine map?
    lin = LinearRegression().fit(Z_hat, Z_true)
    Z_pred = lin.predict(Z_hat)
    z_r2 = r2_score(Z_true, Z_pred, multioutput="variance_weighted")
    z_mse = mean_squared_error(Z_true, Z_pred)

    results.append({
        "seed": s,
        "test_recon_mse": float(test_mse),
        "latent_affine_r2": float(z_r2),
        "latent_affine_mse": float(z_mse),
    })

    trained[s] = {
        "manager": manager,
        "model": shred,
        "engine": engine,
        "Z_hat": Z_hat,
        "lin": lin,
    }

results_df = pd.DataFrame(results).sort_values("seed").reset_index(drop=True)
results_df


## Inspect discovered latent SINDy equations for one run

In [ ]:

seed_to_show = train_seeds[0]
print(f"Seed {seed_to_show} latent forecaster equations:")
print(trained[seed_to_show]["model"].latent_forecaster)



## Notes

- This uses **non-nested** data generation (`nested=False`) and a **4-layer** mixer (`n_layers=4`), as requested.
- PySHRED bottleneck is **3D** via `GRU(hidden_size=3)`.
- The `latent_affine_r2` metric checks whether learned latents are recoverable up to an affine map to true Lorenz coordinates.
- The first code cell auto-installs missing `setuptools` and `pysindy` if needed, which fixes `ModuleNotFoundError: pkg_resources` in fresh environments.
